# DAM Description Extraction Worker 8/12
Deterministic distributed worker 8 of 12 for video-level object descriptions with atomic rclone sync.

In [ ]:
# 1. Wipe any old cached repo and clone fresh from GitHub
!rm -rf /kaggle/working/AIC-2026
!git clone -b feature/dam-text-extraction https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git /kaggle/working/AIC-2026
%cd /kaggle/working/AIC-2026
!git log -1 --oneline

In [ ]:
import os, re, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Install rclone binary
!apt-get update -qq && apt-get install -y -qq rclone

# 2. Write rclone config to all canonical paths & set RCLONE_CONFIG env var
try:
    raw_secret = UserSecretsClient().get_secret('RCLONE_CONFIG_GDRIVE').strip()
    if not raw_secret.startswith('['):
        raw_secret = '[gdrive] ' + raw_secret
    formatted = raw_secret
    if '\n' not in formatted:
        formatted = formatted.replace('[gdrive]', '[gdrive]\n')
        formatted = re.sub(r'\s+(type\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(scope\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(token\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(client_id\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(client_secret\s*=)', r'\n\1', formatted)
    config_paths = [
        '/root/.config/rclone/rclone.conf',
        '/root/.rclone.conf',
        '/kaggle/working/.rclone.conf',
        '/tmp/rclone.conf',
    ]
    for path_str in config_paths:
        p = Path(path_str)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(formatted, encoding='utf-8')
    os.environ['RCLONE_CONFIG'] = '/root/.config/rclone/rclone.conf'
    print('✓ rclone.conf written and RCLONE_CONFIG exported!')
    t = subprocess.run(['rclone', 'listremotes', '--config', '/root/.config/rclone/rclone.conf'], capture_output=True, text=True)
    print('✓ Configured remotes:', t.stdout.strip())
except Exception as exc:
    print(f'❌ Error: {exc}')

In [ ]:
!python -m pip install --quiet --no-deps -r requirements/kaggle.txt

In [ ]:
WORKER_ID = 8
NUM_WORKERS = 12
KEYFRAMES_ROOT = '/kaggle/input/datasets/lyduchoang/aic-26-video/Keyframes/Keyframes'
OBJECTS_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/objects'
MAP_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes'
RCLONE_DEST = 'gdrive:AIC_HCM/artifacts/dam_descriptions/'

# 1-second dry run verification
!python scripts/run_dam_batch.py \
  --worker-id {WORKER_ID} \
  --num-workers {NUM_WORKERS} \
  --keyframes-root {KEYFRAMES_ROOT} \
  --objects-root {OBJECTS_ROOT} \
  --map-keyframes-root {MAP_ROOT} \
  --rclone-dest {RCLONE_DEST} \
  --dry-run

In [ ]:
WORKER_ID = 8
NUM_WORKERS = 12
KEYFRAMES_ROOT = '/kaggle/input/datasets/lyduchoang/aic-26-video/Keyframes/Keyframes'
OBJECTS_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/objects'
MAP_ROOT = '/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes'
RCLONE_DEST = 'gdrive:AIC_HCM/artifacts/dam_descriptions/'

# Launch full automated batch run
!python scripts/run_dam_batch.py \
  --worker-id {WORKER_ID} \
  --num-workers {NUM_WORKERS} \
  --keyframes-root {KEYFRAMES_ROOT} \
  --objects-root {OBJECTS_ROOT} \
  --map-keyframes-root {MAP_ROOT} \
  --rclone-dest {RCLONE_DEST} \
  --output-root /kaggle/working/aic2026-artifacts \
  --cache-root /kaggle/working/aic2026-model-cache \
  --device cuda